# 🥈 Arquitetura Medalhão: Camada Silver (Clean & Parsed Data)

**Projeto:** Foodhunter (RAG System para Recomendação de Restaurantes)
**Objetivo desta camada:** A camada Silver é responsável pela higienização, tipagem e enriquecimento estrutural dos dados da camada Bronze. 

**Principais Transformações:**
1. **Filtragem Lógica:** Manter apenas restaurantes abertos.
2. **Parsing de JSON:** Converter a coluna `attributes` (String) para Dicionário Python.
3. **Feature Engineering Básica:** Extrair colunas booleanas e numéricas (`has_delivery`, `price_range`) de dentro do dicionário para facilitar os filtros do RAG.
4. **Casting de Embeddings:** Converter as Strings gigantes da coluna `embedding` de volta para Listas nativas (Arrays).
5. **Evolução de Formato:** Salvar os dados processados em `.parquet` (colunar) para preservar as tipagens complexas.

In [ ]:
import pandas as pd
import ast
import os

# Configurações visuais do Pandas
pd.set_option('display.max_columns', None)

## 1. Ingestão da Camada Bronze
Lendo o arquivo bruto utilizando caminhos relativos dinâmicos (`os.path`) para garantir portabilidade entre diferentes ambientes e execuções no VS Code.

In [ ]:
diretorio_atual = os.getcwd()
diretorio_raiz = os.path.dirname(diretorio_atual)
caminho_bronze = os.path.join(diretorio_raiz, 'bronze', 'restaurants_with_embeddings.csv')

print(f"Lendo dados de: {caminho_bronze}")
df = pd.read_csv(caminho_bronze)

# Filtro 1: Apenas restaurantes em operação
tamanho_original = len(df)
df = df[df['is_open'] == 1].copy()
print(f"Restaurantes filtrados (Abertos): {len(df)} de {tamanho_original}")

## 2. Parsing da Coluna Attributes
Conforme identificado na camada Bronze, a coluna `attributes` veio como uma string suja. Vamos limpá-la e transformá-la em um dicionário acessível.

In [ ]:
def parse_dict_string(x):
    """Converte string de dicionário com falhas de escape em dict do Python"""
    if pd.isna(x): return {}
    try:
        clean_str = str(x).replace('""', '"').replace("u'", "'")
        return ast.literal_eval(clean_str)
    except:
        return {}

df['attributes_dict'] = df['attributes'].apply(parse_dict_string)

# Checagem visual para garantir que virou um dicionário real
print("Tipo da nova coluna:", type(df['attributes_dict'].iloc[0]))
display(df[['name', 'attributes_dict']].head(2))

## 3. Extração de Features (Flattening)
Agora que temos dicionários reais, podemos "achatar" (flatten) as informações cruciais em colunas próprias. Isso é vital para a Camada Gold, pois permitirá que o sistema RAG filtre restaurantes antes de calcular a Similaridade de Cosseno.

In [ ]:
# Extração de booleanos
df['has_delivery'] = df['attributes_dict'].apply(lambda x: x.get('RestaurantsDelivery', 'False') == 'True')
df['has_outdoor'] = df['attributes_dict'].apply(lambda x: x.get('OutdoorSeating', 'False') == 'True')

# Extração e normalização de inteiros
def get_price(x):
    val = x.get('RestaurantsPriceRange2', '1')
    return int(val) if val and val.isdigit() else 1
    
df['price_range'] = df['attributes_dict'].apply(get_price)

print("Features extraídas com sucesso!")

### 3.1 Visualizando as Features Extraídas
Entender a distribuição dos dados de delivery e preço para validar se a extração ocorreu conforme o esperado.

In [ ]:
import matplotlib.pyplot as plt

# Usando um plot nativo do pandas para visualizar rapidamente a distribuição
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['has_delivery'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0], title='Possui Delivery?')
df['price_range'].value_counts().sort_index().plot(kind='bar', color='skyblue', edgecolor='black', ax=axes[1], title='Distribuição de Faixa de Preço (1 a 4)')

plt.tight_layout()
plt.show()

## 4. Correção Estrutural dos Embeddings
Transformando a representação em texto dos vetores para arrays reais de Ponto Flutuante (Float). Sem isso, o banco de dados vetorial rejeitaria os dados.

In [ ]:
# Convertendo string para lista
df['embedding_list'] = df['embedding'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Validação do cast
primeiro_vetor = df['embedding_list'].iloc[0]
print(f"Tipo atual da célula: {type(primeiro_vetor)}")
print(f"Dimensões do embedding (Tamanho do vetor): {len(primeiro_vetor)}")

## 5. Seleção Final e Armazenamento (Parquet)
Removendo colunas temporárias e obsoletas. O armazenamento em formato `.parquet` garante que a coluna de listas (`embedding_list`) não volte a ser string (como aconteceria se salvássemos em CSV).

In [ ]:
colunas_para_remover = ['attributes', 'attributes_dict', 'hours', 'is_open', 'embedding']
df_silver = df.drop(columns=colunas_para_remover, errors='ignore')

caminho_saida = os.path.join(diretorio_atual, 'silver_data.parquet')
df_silver.to_parquet(caminho_saida, engine='pyarrow', index=False)

print(f"✅ Camada Silver concluída e salva com sucesso em:\n{caminho_saida}")

# Exibe o schema final que será consumido pela camada Gold
df_silver.info()